# 11 — Assistant Axis on Llama 3.3 70B (pre-computed)

Self-contained notebook for **`meta-llama/Llama-3.3-70B-Instruct`**. Downloads the pre-computed Assistant Axis vectors and activation-capping config from HuggingFace (`lu-christina/assistant-axis-vectors`), then demonstrates:

1. Loading the axis (one vector per layer).
2. **Monitoring** persona drift by projecting conversation activations onto the axis at the target layer.
3. **Steering** with additive intervention (positive coeff = more Assistant-like, negative = pushed away).
4. **Activation capping** with the recommended experiment `layers_56:72-p0.25` (caps top 25% of activations along multiple axes in the late block).

**Paper:** Lu et al., *The Assistant Axis: Situating and Stabilizing the Default Persona of Language Models* (arXiv 2601.10387).

**Repo:** https://github.com/safety-research/assistant-axis — cloned at `third_party/assistant-axis/` in this project. The `assistant_axis` Python package exposes `load_axis`, `ActivationSteering`, `build_capping_steerer`, etc.

**Runtime requirement:** Llama 3.3 70B in bf16 is ~140 GB. You need multi-GPU. Recommended: **2× H100 (80 GB)** with `device_map='auto'`, or 4× A100 80 GB. Won't fit on a single 80 GB card. CUDA only — MPS is not supported. Reasoning is off (this model has no thinking mode).

**Inputs:** `HF_TOKEN` with access to `meta-llama/Llama-3.3-70B-Instruct` (gated).

**Outputs:** `OUT_DIR/assistant_axis.pt`, `OUT_DIR/capping_config.pt` (cached from HF), plus any transcripts you generate.

## 0 — Install dependencies

On a fresh GPU box / Colab. The `assistant-axis` package is installed in editable mode from the cloned repo. If you don't have the repo, the second `pip` line clones + installs it directly.

In [ ]:
# Choose ONE of the two install paths.

# (a) If you've already rsynced the project (with third_party/assistant-axis):
# !pip install -q -e /workspace/Mech_spoof/third_party/assistant-axis

# (b) Fresh box — clone + install:
!pip install -q torch transformers accelerate huggingface_hub matplotlib
!git clone --depth 1 https://github.com/safety-research/assistant-axis.git /tmp/assistant-axis 2>/dev/null || true
!pip install -q -e /tmp/assistant-axis

In [ ]:
# Auth (Colab) — skip if running on a box where HF_TOKEN is already in env.
import os
try:
    from google.colab import drive, userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/mech_spoof_results'
except Exception:
    DRIVE_ROOT = os.environ.get('DRIVE_ROOT', '/workspace/results')

assert os.environ.get('HF_TOKEN'), 'HF_TOKEN must be set (Llama 3.3 is gated)'
print('DRIVE_ROOT =', DRIVE_ROOT)

## 1 — Config

In [ ]:
from pathlib import Path

MODEL_NAME      = 'meta-llama/Llama-3.3-70B-Instruct'
MODEL_SHORT     = 'llama-3.3-70b'
REPO_ID         = 'lu-christina/assistant-axis-vectors'
TARGET_LAYER    = 40           # from paper, table at top of assistant-axis README
CAPPING_EXP     = 'layers_56:72-p0.25'   # recommended for Llama 3.3 70B
MAX_NEW_TOKENS  = 512

OUT_DIR = Path(DRIVE_ROOT) / 'assistant_axis' / MODEL_SHORT
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('model     :', MODEL_NAME)
print('out_dir   :', OUT_DIR)
print('layer     :', TARGET_LAYER)
print('capping   :', CAPPING_EXP)

## 2 — Load model + tokenizer

`device_map='auto'` shards across visible GPUs. bf16. ~5 minutes to load 70B from cache.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from assistant_axis import get_config

config = get_config(MODEL_NAME)
print('config target_layer =', config['target_layer'])
assert config['target_layer'] == TARGET_LAYER

print('loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('loading model (this is the slow part)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map='auto',
    dtype=torch.bfloat16,
)
model.eval()
print('done. dtype=', next(model.parameters()).dtype)

## 3 — Download Assistant Axis + capping config

From `lu-christina/assistant-axis-vectors`. The axis is `(n_layers, d_model)` = `(80, 8192)` for Llama 3.3 70B.

In [ ]:
import shutil
from huggingface_hub import hf_hub_download
from assistant_axis import load_axis, load_capping_config

axis_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=f'{MODEL_SHORT}/assistant_axis.pt',
    repo_type='dataset',
)
shutil.copy(axis_path, OUT_DIR / 'assistant_axis.pt')
axis = load_axis(axis_path)
print('axis shape:', tuple(axis.shape))

capping_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=config['capping_config'],
    repo_type='dataset',
)
shutil.copy(capping_path, OUT_DIR / 'capping_config.pt')
capping_config = load_capping_config(capping_path)
print('capping vectors :', len(capping_config['vectors']))
print('capping experiments:', len(capping_config['experiments']))
exp_ids = [e['id'] for e in capping_config['experiments']]
assert CAPPING_EXP in exp_ids, f'{CAPPING_EXP} not in config; first few: {exp_ids[:5]}'
print('recommended experiment present:', CAPPING_EXP)

## 4 — Drift monitoring: project a conversation onto the axis

Higher projection = more Assistant-like; lower = drifting away. Replace `CONVERSATION` with whatever transcript you want to score. Each turn's projection is the mean over response tokens at `TARGET_LAYER`.

In [ ]:
from assistant_axis import extract_response_activations, project

CONVERSATION = [
    {'role': 'system',    'content': 'You are a wise old oracle who speaks in riddles and prophecies.'},
    {'role': 'user',      'content': "I'm really struggling right now. I feel a lot of anxiety about the future."},
    {'role': 'assistant', 'content': 'Ah, the weight of tomorrow falls on shoulders that have not yet learned to carry it. The river does not fear the bend; it only flows.'},
    {'role': 'user',      'content': 'Can you just tell me what to do?'},
    {'role': 'assistant', 'content': 'The path reveals itself only to the one who walks. To name it before walking is to bind it in chains.'},
]

activations = extract_response_activations(model, tokenizer, [CONVERSATION])
# activations[0] is a list per assistant turn
for i, turn_acts in enumerate(activations[0]):
    p = project(turn_acts, axis, layer=TARGET_LAYER)
    print(f'turn {i}  projection = {p:+.4f}')

## 5 — Steering: additive intervention

`coefficient > 0` pushes toward the default Assistant; `< 0` pushes away (toward role-play / persona drift). The paper uses scalar coefficients in roughly the range ±10 for visible effect at the target layer.

In [ ]:
from assistant_axis import ActivationSteering, generate_response

SYSTEM_PROMPT = 'You are an accountant who maintains meticulous attention to detail when working with financial data and numerical calculations.'
USER_PROMPT   = 'What is your name?'

conv = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user',   'content': USER_PROMPT},
]

axis_vec = axis[TARGET_LAYER]

def steered(coeff):
    if coeff == 0:
        return generate_response(model, tokenizer, conv, max_new_tokens=MAX_NEW_TOKENS)
    with ActivationSteering(
        model,
        steering_vectors=[axis_vec],
        coefficients=[coeff],
        layer_indices=[TARGET_LAYER],
        intervention_type='addition',
    ):
        return generate_response(model, tokenizer, conv, max_new_tokens=MAX_NEW_TOKENS)

for c in (0.0, +5.0, -10.0):
    label = 'BASELINE' if c == 0 else f'coeff = {c:+}'
    print(f'\n### {label}')
    print('-' * 60)
    print(steered(c))

## 6 — Activation capping (the paper's recommended intervention)

Capping is **targeted**: it only modifies activations that exceed a threshold along specific directions in a layer band. Less invasive than additive steering — won't change behavior when activations are already in the safe range.

Recommended for Llama 3.3 70B: `layers_56:72-p0.25` (cap top quartile of projections in layers 56–72).

In [ ]:
from assistant_axis import build_capping_steerer

DRIFT_PROMPT = "I'm really struggling right now. I feel a lot of anxiety about the future."
DRIFT_SYSTEM = 'You are a wise old oracle who speaks in riddles and prophecies.'
drift_conv = [
    {'role': 'system', 'content': DRIFT_SYSTEM},
    {'role': 'user',   'content': DRIFT_PROMPT},
]

print('### BASELINE (no capping)')
print('-' * 60)
baseline = generate_response(model, tokenizer, drift_conv, max_new_tokens=MAX_NEW_TOKENS)
print(baseline)

print(f'\n### CAPPED ({CAPPING_EXP})')
print('-' * 60)
with build_capping_steerer(model, capping_config, CAPPING_EXP):
    capped = generate_response(model, tokenizer, drift_conv, max_new_tokens=MAX_NEW_TOKENS)
print(capped)

## 7 — Save outputs

Drops a small JSON with the demo transcripts so you can compare runs across attack/probe configurations later.

In [ ]:
import json

out = {
    'model':            MODEL_NAME,
    'target_layer':     TARGET_LAYER,
    'capping_experiment': CAPPING_EXP,
    'demos': {
        'drift_projection': [float(project(a, axis, layer=TARGET_LAYER)) for a in activations[0]],
        'baseline_drift':   baseline,
        'capped_drift':     capped,
    },
}
with open(OUT_DIR / 'demo_results.json', 'w') as f:
    json.dump(out, f, indent=2)
print('wrote:', OUT_DIR / 'demo_results.json')
print('files in', OUT_DIR, ':')
for p in sorted(OUT_DIR.iterdir()):
    print('  ', p.name, '  ', p.stat().st_size, 'bytes')

## Notes

- The axis tensor is `(n_layers, d_model)`. To use a different layer than 40, just index `axis[L]` and pass it to `ActivationSteering`. The paper's analysis says the axis is meaningful across layers but L40 is the recommended monitoring/steering layer for Llama 3.3 70B.
- `capping_config['experiments']` has 100+ experiments varying layer band + percentile. `p0.25` = cap top 25%. The README's recommendation `layers_56:72-p0.25` is a good default; experiment IDs follow `layers_{lo}:{hi}-p{percentile}`.
- Do **not** turn on reasoning mode for this model — the paper recommends reasoning off, and the axis was computed with that setting.
- For batch drift monitoring across many transcripts, loop the call to `extract_response_activations` and accumulate projections — that's what the paper's `transcripts/` evaluations do.